In [0]:
import os , sys

current = os.getcwd()
reusable = os.path.join(current , ".." , "..")
print(reusable)
sys.path.append(reusable)
from utils.modules import *


In [0]:
%fs ls "abfss://bronze@spotifyproject.dfs.core.windows.net"

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimUser/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@spotifyproject.dfs.core.windows.net/DimUser/*")

In [0]:
df_transformed = df_user.withColumn("user_name",upper(col("user_name")))

In [0]:
obj = Transform()
df_transformed = obj.dropColumn(df_transformed,["_rescued_data"])


In [0]:
df_transformed = df_transformed.dropDuplicates(['user_id'])

In [0]:
df_transformed.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimUser/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@spotifyproject.dfs.core.windows.net/DimUser/data")\
            .toTable("spotify_catalog.silver.DimUser")

In [0]:
%sql
select * from spotify_catalog.silver.DimUser